# Simple FastAPI — Beginner's Guide
### Covers: Routers · App Context · Rate Limiting · CRUD · Search

---

## What are we building?
A small **Shop API** that manages grocery items.  
Any app (website, mobile app) can call it to get, add, update or delete items.

```
Your App  ──►  FastAPI  ──►  Items stored in memory
          ◄──           ◄──  (like a Python dictionary)
```

## Concepts covered
| Concept | Plain English |
|---|---|
| **Router** | A way to group related routes into separate files/sections |
| **App Context** | Shared information about the app (name, version, owner) |
| **Rate Limiting** | Limiting how many times someone can call an endpoint per minute |
| **Schema (Pydantic)** | Rules for what data must look like |
| **CRUD** | Create · Read · Update · Delete — the 4 basic operations |
| **Search** | Filter items by category or price |

---
⚠️ Run cells **top to bottom** in order.

## Cell 1 — Install Packages

In [1]:
import subprocess, sys

# These are the only packages we need for this simple API
packages = [
    "fastapi",    # The web framework
    "uvicorn",    # The server that runs FastAPI
    "pydantic",   # Data validation
    "slowapi",    # Rate limiting
    "httpx",      # Needed by TestClient to make test requests
]

# Try normal install, fall back for Ubuntu/Debian systems
base = [sys.executable, "-m", "pip", "install", "--quiet"]
result = subprocess.run(base + packages, capture_output=True)
if result.returncode != 0:
    subprocess.check_call(base + ["--break-system-packages"] + packages)

print("Packages installed!")

Packages installed!


---
## Cell 2 — The "Database" (just a Python dictionary)

In this beginner example we skip a real database.  
We just use a Python **dictionary** — it works exactly the same way for learning.  
Think of it as a spreadsheet in memory.

```
items_db = {
   1 → { id:1, name:"Apple",  price:30, category:"Fruit"  }
   2 → { id:2, name:"Milk",   price:55, category:"Dairy"  }
   3 → { id:3, name:"Bread",  price:40, category:"Bakery" }
}
```
The **key** is the item ID. The **value** is all the item's details.

In [2]:
# Our in-memory "database" — a plain Python dictionary
# WHY dict? Because looking up by ID (items_db[1]) is instant.
# WHY not a list? Lists need .index() to search — slow. Dict lookup is O(1).
items_db = {
    1: {"id": 1, "name": "Apple",  "price": 30.0,  "category": "Fruit"},
    2: {"id": 2, "name": "Milk",   "price": 55.0,  "category": "Dairy"},
    3: {"id": 3, "name": "Bread",  "price": 40.0,  "category": "Bakery"},
    4: {"id": 4, "name": "Butter", "price": 85.0,  "category": "Dairy"},
    5: {"id": 5, "name": "Orange", "price": 25.0,  "category": "Fruit"},
}

# This counter tracks the next ID to assign when a new item is created
# WHY 'global'? Because it lives outside any function but needs to be changed inside one
next_id = 6

print("Sample data loaded:")
for item in items_db.values():
    print(f"  id={item['id']}  ₹{item['price']:<6}  {item['category']:<8}  {item['name']}")

Sample data loaded:
  id=1  ₹30.0    Fruit     Apple
  id=2  ₹55.0    Dairy     Milk
  id=3  ₹40.0    Bakery    Bread
  id=4  ₹85.0    Dairy     Butter
  id=5  ₹25.0    Fruit     Orange


---
## Cell 3 — Pydantic Schemas (The Rules)

A schema says: **"This is what the data must look like."**

Without schemas, someone could send `price = -500` or `name = ""`.  
Pydantic **automatically rejects** bad data before your code even runs.

```
Client sends →  Pydantic checks it  →  If bad: return error immediately
                                    →  If good: pass it to your function
```

In [3]:
from pydantic import BaseModel, Field
from typing import Optional, List

# Schema for CREATING an item (what the client sends TO us)
class ItemCreate(BaseModel):
    # Field(...) means REQUIRED — must be provided
    # min_length=2 means name cannot be 1 character like "A"
    name: str     = Field(..., min_length=2, example="Mango")

    # gt=0 means 'greater than 0' — price must be positive
    price: float  = Field(..., gt=0, example=45.0)

    category: str = Field(..., example="Fruit")


# Schema for RESPONSES (what we send BACK to the client)
# Same as ItemCreate but includes 'id' — the database assigns the ID,
# so the client doesn't send it, but they receive it back
class ItemResponse(BaseModel):
    id: int
    name: str
    price: float
    category: str


# ── Quick test ────────────────────────────────────────────────────────────────
from pydantic import ValidationError

# Valid item — passes
good = ItemCreate(name="Mango", price=45.0, category="Fruit")
print("Valid item:", good)

# Invalid item — Pydantic catches it
try:
    bad = ItemCreate(name="A", price=-10, category="Fruit")  # name too short, price negative
except ValidationError as e:
    print("\nValidation errors caught (expected):")
    for err in e.errors():
        print(f"  field={err['loc']}  →  {err['msg']}")

Valid item: name='Mango' price=45.0 category='Fruit'

Validation errors caught (expected):
  field=('name',)  →  String should have at least 2 characters
  field=('price',)  →  Input should be greater than 0


/var/folders/hq/v_7wt_d51sn37zqs9vlsf41c0000gn/T/ipykernel_30775/2064059817.py:8: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  name: str     = Field(..., min_length=2, example="Mango")
/var/folders/hq/v_7wt_d51sn37zqs9vlsf41c0000gn/T/ipykernel_30775/2064059817.py:11: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  price: float  = Field(..., gt=0, example=45.0)
/var/folders/hq/v_7wt_d51sn37zqs9vlsf41c0000gn/T/ipykernel_30775/2064059817.py:13: PydanticDeprecatedSince20: Using extra keyword arguments on `

---
## Cell 4 — App Context

**App Context** = a dictionary of shared information about your app.  
Things like the app name, version, and owner that any part of the app can access.

It's exposed via the `/info` route so anyone calling the API can see what they're talking to.

In [4]:
# App context — shared info about this API
# WHY a dict? Simple key-value store, easy to add new fields later
APP_CONTEXT = {
    "app_name":    "My Simple Shop API",
    "version":     "1.0",
    "owner":       "Datasphere",
    "description": "A beginner-friendly grocery shop API",
    "contact":     "support@datasphere.ai",
}

print("App context ready:")
for key, value in APP_CONTEXT.items():
    print(f"  {key}: {value}")

App context ready:
  app_name: My Simple Shop API
  version: 1.0
  owner: Datasphere
  description: A beginner-friendly grocery shop API
  contact: support@datasphere.ai


---
## Cell 5 — Rate Limiter Setup

**Rate limiting** = controlling how many times someone can call your API per minute.

```
Without rate limiting:  One person can send 10,000 requests/second → your server crashes
With rate limiting:     Each person gets max 10 requests/minute → fair and safe
```

We use the **client's IP address** to identify who is making the requests.

In [5]:
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded

# get_remote_address is a function that reads the client's IP from each request
# The limiter uses this IP to count requests per person
limiter = Limiter(key_func=get_remote_address)

print("Rate limiter created.")
print("It uses the client's IP address to track request counts.")
print("Example limits we will use:")
print("  GET  /items   →  10 requests per minute (reading is cheap)")
print("  POST /items   →   5 requests per minute (writing costs more)")

Rate limiter created.
It uses the client's IP address to track request counts.
Example limits we will use:
  GET  /items   →  10 requests per minute (reading is cheap)
  POST /items   →   5 requests per minute (writing costs more)


---
## Cell 6 — Create the App + Routers

### What is a Router?

Imagine a big restaurant. Instead of one menu for everything, you have:
- A **food menu** (items routes)
- An **info card** (info routes)

A **Router** groups related routes together, just like sections on a menu.  
In a real project, each router would be in its own file (`items.py`, `orders.py`, etc.).

```
app
 ├── info_router  →  /info/          (app info)
 │                   /info/health    (health check)
 │
 └── items_router →  /items/         (list all)
                     /items/{id}     (get one)
                     /items/search/  (search)
                     POST /items/    (create)
                     PATCH /items/{id} (update)
                     DELETE /items/{id} (delete)
```

In [6]:
from fastapi import FastAPI, APIRouter, HTTPException, Request, Query, Path, Body

# ── Create the main app ───────────────────────────────────────────────────────
app = FastAPI(
    title="My Simple Shop API",   # Shows in Swagger docs
    version="1.0",
    description="A beginner-friendly grocery shop API"
)

# Attach the rate limiter to the app
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)


# ── Create two routers ────────────────────────────────────────────────────────

# Router 1: Info routes
# prefix="/info" means ALL routes in this router start with /info
# tags=["Info"] groups them under "Info" in Swagger docs
info_router = APIRouter(prefix="/info", tags=["Info"])

# Router 2: Items routes
# prefix="/items" means ALL routes start with /items
items_router = APIRouter(prefix="/items", tags=["Items"])


print("App and routers created.")
print("  info_router  →  handles /info/...  routes")
print("  items_router →  handles /items/... routes")

App and routers created.
  info_router  →  handles /info/...  routes
  items_router →  handles /items/... routes


---
## Cell 7 — Info Router Routes

Two simple routes that return information about the API itself.

In [7]:
# ── Route 1: GET /info/ ───────────────────────────────────────────────────────
# Returns everything in our APP_CONTEXT dict
@info_router.get("/")
def get_app_info():
    """
    Returns general information about this API.
    Useful for clients to know what version they are talking to.
    """
    return APP_CONTEXT   # FastAPI auto-converts this dict to JSON


# ── Route 2: GET /info/health ─────────────────────────────────────────────────
# A quick ping to check the API is alive
# Docker and load balancers call this every 30 seconds
@info_router.get("/health")
def health_check():
    """
    Simple health check — returns 200 OK if the server is running.
    Used by Docker to detect if the container has crashed.
    """
    return {
        "status": "ok",
        "app":    APP_CONTEXT["app_name"],
        "total_items_in_store": len(items_db)  # Live count from our "database"
    }


print("Info routes defined:")
print("  GET /info/         →  returns app name, version, owner")
print("  GET /info/health   →  returns status: ok")

Info routes defined:
  GET /info/         →  returns app name, version, owner
  GET /info/health   →  returns status: ok


---
## Cell 8 — Items Router Routes (CRUD + Search)

**CRUD** = the 4 things you can do to any data:

| Letter | Action | HTTP Method | Example |
|---|---|---|---|
| **C** | Create | POST | Add a new item |
| **R** | Read | GET | Get item(s) |
| **U** | Update | PATCH | Change the price |
| **D** | Delete | DELETE | Remove an item |

In [8]:
# ── READ: GET /items/ — List all items ───────────────────────────────────────
@items_router.get("/", response_model=List[ItemResponse])
@limiter.limit("10/minute")   # Max 10 requests per minute per IP
def list_all_items(request: Request):
    """
    Returns every item in the store.
    Rate limited to 10 per minute — reading is cheap but still needs a limit.
    """
    # .values() gives us all the dicts in items_db
    # list() converts the dict_values into a proper list
    return list(items_db.values())


# ── READ: GET /items/search/ — Search and filter ─────────────────────────────
# IMPORTANT: /search/ must come BEFORE /{item_id}
# If /{item_id} was first, FastAPI would try to cast "search" as an integer → error
@items_router.get("/search/", response_model=List[ItemResponse])
def search_items(
    # Query parameters appear in the URL after '?'
    # Example: /items/search/?category=Fruit&max_price=50
    category:  Optional[str]   = Query(None, description="Filter by category, e.g. Fruit"),
    max_price: Optional[float] = Query(None, gt=0, description="Only show items cheaper than this"),
):
    """
    Search items by category and/or price.
    Both filters are optional — leave them out to get everything.
    """
    results = list(items_db.values())  # Start with all items

    # Only apply the filter if the client actually sent it
    if category:
        # Keep only items whose category matches (case-sensitive)
        results = [i for i in results if i["category"] == category]

    if max_price is not None:
        # Keep only items cheaper than or equal to max_price
        results = [i for i in results if i["price"] <= max_price]

    return results


# ── READ: GET /items/{item_id} — Get one specific item ───────────────────────
@items_router.get("/{item_id}", response_model=ItemResponse)
def get_one_item(
    # Path(..., ge=1) means this URL param must be an integer >= 1
    # /items/0 or /items/-5 will be auto-rejected with 422
    item_id: int = Path(..., ge=1, description="The ID of the item")
):
    """
    Returns one specific item by its ID.
    Returns 404 if the item doesn't exist.
    """
    # Check if this ID exists in our dictionary
    if item_id not in items_db:
        # HTTPException stops execution here and sends an error response
        # status_code=404 means 'Not Found'
        raise HTTPException(
            status_code=404,
            detail=f"Item with id={item_id} does not exist"
        )
    return items_db[item_id]


# ── CREATE: POST /items/ — Add a new item ─────────────────────────────────────
@items_router.post("/", response_model=ItemResponse, status_code=201)
# status_code=201 = 'Created' — the correct HTTP code when something new is made
@limiter.limit("5/minute")   # Stricter limit for writes — creating data is more expensive
def create_item(
    request: Request,   # Required by the rate limiter to read the client's IP
    item: ItemCreate,   # FastAPI reads the JSON body and validates it against ItemCreate
):
    """
    Creates a new item in the store.
    Rate limited to 5 per minute — writing is more expensive than reading.
    """
    global next_id  # We need to modify this variable that lives outside the function

    # Build the new item dict
    # item.model_dump() converts the Pydantic object → plain dict
    # e.g. ItemCreate(name='Mango', price=45, category='Fruit')
    #    → {'name': 'Mango', 'price': 45, 'category': 'Fruit'}
    new_item = {
        "id": next_id,    # Assign the next available ID
        **item.model_dump()  # Unpack name, price, category into the dict
    }

    items_db[next_id] = new_item  # Save to our "database"
    next_id += 1                  # Increment so the next item gets a different ID

    return new_item


# ── UPDATE: PATCH /items/{item_id} — Change specific fields ──────────────────
@items_router.patch("/{item_id}", response_model=ItemResponse)
# PATCH = partial update (only send the fields you want to change)
# PUT   = full replace   (you must send ALL fields every time)
def update_item(
    item_id: int,
    updates: dict = Body(...)  # Accept any dict — client sends only what they want to change
    # WHY dict not a Pydantic model? A model would require all fields.
    # dict lets the client send just {"price": 35} without touching name or category.
):
    """
    Partially updates an item — only the fields you send get changed.
    Example: send {"price": 35} to only update the price.
    """
    if item_id not in items_db:
        raise HTTPException(status_code=404, detail=f"Item {item_id} not found")

    # Only update fields that actually exist on the item
    # This prevents the client from adding weird keys like {"hacked": True}
    for key, value in updates.items():
        if key in items_db[item_id] and key != "id":  # Don't allow changing the ID!
            items_db[item_id][key] = value

    return items_db[item_id]


# ── DELETE: DELETE /items/{item_id} — Remove an item ─────────────────────────
@items_router.delete("/{item_id}", status_code=204)
# status_code=204 = 'No Content' — correct code for DELETE (nothing to return)
def delete_item(item_id: int):
    """
    Permanently removes an item from the store.
    Returns nothing (204 No Content).
    """
    if item_id not in items_db:
        raise HTTPException(status_code=404, detail=f"Item {item_id} not found")

    del items_db[item_id]  # Remove the key from our dictionary — permanently gone
    # No return statement needed — FastAPI sends 204 with empty body automatically


print("Items routes defined:")
print("  GET    /items/           →  list all (rate: 10/min)")
print("  GET    /items/search/    →  search by category / price")
print("  GET    /items/{id}       →  get one item")
print("  POST   /items/           →  create new item (rate: 5/min)")
print("  PATCH  /items/{id}       →  update specific fields")
print("  DELETE /items/{id}       →  delete item")

Items routes defined:
  GET    /items/           →  list all (rate: 10/min)
  GET    /items/search/    →  search by category / price
  GET    /items/{id}       →  get one item
  POST   /items/           →  create new item (rate: 5/min)
  PATCH  /items/{id}       →  update specific fields
  DELETE /items/{id}       →  delete item


---
## Cell 9 — Register Routers with the App

The routers are defined but not connected yet.  
`app.include_router()` plugs them into the main app — like connecting a branch to a tree.

In [9]:
# Connect the routers to the main app
# After this, all routes in each router are active
app.include_router(items_router)   # Adds all /items/... routes
app.include_router(info_router)    # Adds all /info/...  routes

# Show all registered routes
print("All routes registered on the app:")
print()
for route in app.routes:
    if hasattr(route, "methods"):   # Skip non-endpoint routes (like middleware)
        methods = ", ".join(route.methods)
        print(f"  {methods:<8}  {route.path}")

All routes registered on the app:

  GET, HEAD  /openapi.json
  GET, HEAD  /docs
  GET, HEAD  /docs/oauth2-redirect
  GET, HEAD  /redoc
  GET       /items/
  GET       /items/search/
  GET       /items/{item_id}
  POST      /items/
  PATCH     /items/{item_id}
  DELETE    /items/{item_id}
  GET       /info/
  GET       /info/health


---
## Cell 10 — Test Every Route

`TestClient` lets us call the API like a real HTTP client — no server needed.  
It's perfect for testing in a notebook.

In [10]:
from fastapi.testclient import TestClient
import json

client = TestClient(app, raise_server_exceptions=False)

def show(label, r):
    """Prints a test result cleanly."""
    try:
        body = r.json()
    except Exception:
        body = r.text or "(empty body)"
    print(f"  [{r.status_code}] {label}")
    print(f"       {body}")
    print()


# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("INFO ROUTES")
print("=" * 60)

show("GET /info/health", client.get("/info/health"))
show("GET /info/",       client.get("/info/"))


# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("READ — List & Get")
print("=" * 60)

r = client.get("/items/")
print(f"  [200] GET /items/  →  {len(r.json())} items returned")
for item in r.json():
    print(f"    id={item['id']}  ₹{item['price']:<6}  {item['category']:<8}  {item['name']}")
print()

show("GET /items/1  (get Apple)",   client.get("/items/1"))
show("GET /items/999 (not found)",  client.get("/items/999"))
show("GET /items/0  (invalid id)",  client.get("/items/0"))   # ge=1 rejects 0


# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("SEARCH — Filter by category and price")
print("=" * 60)

r = client.get("/items/search/?category=Fruit")
print(f"  [200] GET /items/search/?category=Fruit  →  {len(r.json())} results")
for item in r.json(): print(f"    {item['name']}  ₹{item['price']}")
print()

r = client.get("/items/search/?max_price=45")
print(f"  [200] GET /items/search/?max_price=45  →  {len(r.json())} results (items ≤ ₹45)")
for item in r.json(): print(f"    {item['name']}  ₹{item['price']}")
print()

r = client.get("/items/search/?category=Dairy&max_price=60")
print(f"  [200] GET /items/search/?category=Dairy&max_price=60  →  {len(r.json())} results")
for item in r.json(): print(f"    {item['name']}  ₹{item['price']}")
print()


# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("CREATE — POST new items")
print("=" * 60)

show("POST /items/ (add Mango)",
     client.post("/items/", json={"name": "Mango", "price": 45.0, "category": "Fruit"}))

show("POST /items/ (add Paneer)",
     client.post("/items/", json={"name": "Paneer", "price": 120.0, "category": "Dairy"}))

# Bad data — Pydantic should reject this (name too short, price negative)
show("POST /items/ (bad data — should fail)",
     client.post("/items/", json={"name": "X", "price": -10, "category": "Fruit"}))


# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("UPDATE — PATCH existing items")
print("=" * 60)

# Only changing the price — name and category stay the same
show("PATCH /items/1 (change Apple price to ₹35)",
     client.patch("/items/1", json={"price": 35.0}))

# Changing both name and category at once
show("PATCH /items/3 (rename Bread to Sourdough)",
     client.patch("/items/3", json={"name": "Sourdough", "category": "Bakery"}))

# Trying to change the ID — should be ignored (our code skips 'id' updates)
show("PATCH /items/1 (try to change id — should be ignored)",
     client.patch("/items/1", json={"id": 999}))


# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("DELETE — Remove items")
print("=" * 60)

r = client.delete("/items/2")
print(f"  [{r.status_code}] DELETE /items/2  →  204 means deleted, no body returned")
print()

show("GET /items/2 after delete (should be 404)", client.get("/items/2"))


# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("FINAL STATE of the store")
print("=" * 60)
r = client.get("/items/")
print(f"  {len(r.json())} items remain in store:")
for item in r.json():
    print(f"    id={item['id']}  ₹{item['price']:<6}  {item['category']:<8}  {item['name']}")

INFO ROUTES
  [200] GET /info/health
       {'status': 'ok', 'app': 'My Simple Shop API', 'total_items_in_store': 5}

  [200] GET /info/
       {'app_name': 'My Simple Shop API', 'version': '1.0', 'owner': 'Datasphere', 'description': 'A beginner-friendly grocery shop API', 'contact': 'support@datasphere.ai'}

READ — List & Get
  [200] GET /items/  →  5 items returned
    id=1  ₹30.0    Fruit     Apple
    id=2  ₹55.0    Dairy     Milk
    id=3  ₹40.0    Bakery    Bread
    id=4  ₹85.0    Dairy     Butter
    id=5  ₹25.0    Fruit     Orange

  [200] GET /items/1  (get Apple)
       {'id': 1, 'name': 'Apple', 'price': 30.0, 'category': 'Fruit'}

  [404] GET /items/999 (not found)
       {'detail': 'Item with id=999 does not exist'}

  [422] GET /items/0  (invalid id)
       {'detail': [{'type': 'greater_than_equal', 'loc': ['path', 'item_id'], 'msg': 'Input should be greater than or equal to 1', 'input': '0', 'ctx': {'ge': 1}}]}

SEARCH — Filter by category and price
  [200] GET /items/

---
## Cell 11 — See Rate Limiting in Action

In [11]:
print("Calling POST /items/ 6 times (limit is 5/minute)...")
print("The 6th call should return HTTP 429 Too Many Requests.")
print()

for i in range(1, 8):
    r = client.post("/items/", json={
        "name": f"TestItem{i}",
        "price": float(10 * i),
        "category": "Test"
    })

    if r.status_code == 201:
        print(f"  Call {i}: ✓ {r.status_code} Created  — item saved")
    elif r.status_code == 429:
        print(f"  Call {i}: ✗ {r.status_code} Too Many Requests — RATE LIMIT HIT")
        print(f"         Response: {r.json()}")
    else:
        print(f"  Call {i}: ? {r.status_code} — {r.text}")

Calling POST /items/ 6 times (limit is 5/minute)...
The 6th call should return HTTP 429 Too Many Requests.

  Call 1: ✓ 201 Created  — item saved
  Call 2: ✓ 201 Created  — item saved
  Call 3: ✓ 201 Created  — item saved
  Call 4: ✗ 429 Too Many Requests — RATE LIMIT HIT
         Response: {'error': 'Rate limit exceeded: 5 per 1 minute'}
  Call 5: ✗ 429 Too Many Requests — RATE LIMIT HIT
         Response: {'error': 'Rate limit exceeded: 5 per 1 minute'}
  Call 6: ✗ 429 Too Many Requests — RATE LIMIT HIT
         Response: {'error': 'Rate limit exceeded: 5 per 1 minute'}
  Call 7: ✗ 429 Too Many Requests — RATE LIMIT HIT
         Response: {'error': 'Rate limit exceeded: 5 per 1 minute'}


---
## Cell 12 — Start the Live Server (optional)

Run this to open the real API in your browser with **interactive Swagger docs**.  
You can try every endpoint visually — no code needed.

⚠️ This **blocks** the kernel. Stop with **Kernel → Interrupt**.

In [12]:
import uvicorn

print("Uncomment the line below and run to start the server.")
print()
print("  Swagger UI  →  http://localhost:8000/docs")
print("  All routes  →  http://localhost:8000/openapi.json")
print()
print("Try these URLs in your browser after starting:")
print("  http://localhost:8000/info/health")
print("  http://localhost:8000/items/")
print("  http://localhost:8000/items/1")
print("  http://localhost:8000/items/search/?category=Fruit")

# Uncomment to start:
# uvicorn.run(app, host="0.0.0.0", port=8000)

Uncomment the line below and run to start the server.

  Swagger UI  →  http://localhost:8000/docs
  All routes  →  http://localhost:8000/openapi.json

Try these URLs in your browser after starting:
  http://localhost:8000/info/health
  http://localhost:8000/items/
  http://localhost:8000/items/1
  http://localhost:8000/items/search/?category=Fruit


---
## Quick Reference — Everything in One Place

### All Routes

| Method | URL | What it does | Rate limit |
|---|---|---|---|
| GET | `/info/` | App name, version, owner | none |
| GET | `/info/health` | Is the API alive? | none |
| GET | `/items/` | List every item | 10/min |
| GET | `/items/search/` | Filter by category and/or price | none |
| GET | `/items/{id}` | Get one specific item | none |
| POST | `/items/` | Create a new item | 5/min |
| PATCH | `/items/{id}` | Change specific fields of an item | none |
| DELETE | `/items/{id}` | Remove an item forever | none |

### HTTP Status Codes Used

| Code | Meaning | When you see it |
|---|---|---|
| 200 | OK | Successful GET or PATCH |
| 201 | Created | Successful POST |
| 204 | No Content | Successful DELETE |
| 404 | Not Found | Item ID doesn't exist |
| 422 | Unprocessable | Pydantic validation failed |
| 429 | Too Many Requests | Rate limit exceeded |

### Key Concepts Summary

```
FastAPI     = the waiter who takes requests and returns responses
Pydantic    = the form that validates data before it reaches the kitchen
Router      = a section of the menu (group of related routes)
App Context = shared info about the app (name, version, owner)
Rate Limit  = max N requests per minute per IP address
TestClient  = lets you call the API in Python without starting a server
Uvicorn     = the server that makes the API accessible on a real URL
```